In [90]:
import numpy as np
import sys
import time
import h5py
from tqdm import tqdm

import numpy as np
import re
from math import ceil
from sklearn.metrics import average_precision_score
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import pickle
#import pickle5 as pickle

from sklearn.model_selection import train_test_split

from scipy.sparse import load_npz
from glob import glob

from transformers import get_constant_schedule_with_warmup
from sklearn.metrics import precision_score,recall_score,accuracy_score
import copy

from src.train import trainModel
#from src.dataloader import getData,spliceDataset,h5pyDataset,collate_fn
from src.dataloader import getData,spliceDataset,h5pyDataset,getDataPointList,getDataPointListFull,DataPointFull
from src.weight_init import keras_init
from src.losses import categorical_crossentropy_2d
from src.model import SpliceFormer
from src.evaluation_metrics import print_topl_statistics,cross_entropy_2d
from src.gpu_metrics import run_bootstrap, calculate_ap, calculate_topk
import os

In [1]:
import gffutils

# Replace with your GTF path
gtf_path = '/Users/douglasflorizone/Downloads/gencode.v39.annotation.gtf'

db_path = '/Users/douglasflorizone/Downloads/gencode_v39.db'

# Create a database (this can take a few minutes the first time)
# db = gffutils.create_db(gtf_path, dbfn=db_path,force=True,disable_infer_genes=True, disable_infer_transcripts=True)

In [23]:
def getJunctions(gtf,transcript_id):
    # transcript = gtf[transcript_id.split('.')[0]]
    transcript = gtf[transcript_id]
    strand = transcript[6]
    exon_junctions = []
    tx_start = int(transcript[3])
    tx_end = int(transcript[4])
    exons = gtf.children(transcript, featuretype="exon")
    for exon in exons:
        exon_start = int(exon[3])
        exon_end = int(exon[4])
        exon_junctions.append((exon_start,exon_end))

    intron_junctions = []

    if strand=='+':
        intron_start = exon_junctions[0][1]
        for i,exon_junction in enumerate(exon_junctions[1:]):
            intron_end = exon_junction[0]
            intron_junctions.append((intron_start,intron_end))
            if i+1 != len(exon_junctions[1:]):
                intron_start = exon_junction[1]

    elif strand=='-':
        exon_junctions.reverse()
        intron_start = exon_junctions[0][1]
        for i,exon_junction in enumerate(exon_junctions[1:]):
            intron_end = exon_junction[0]
            intron_junctions.append((intron_start,intron_end))
            if i+1 != len(exon_junctions[1:]):
                intron_start = exon_junction[1]

    jn_start = [x[0] for x in intron_junctions]
    jn_end = [x[1] for x in intron_junctions]
    Y_type, Y_idx = [],[]
    if strand == '+':
        Y0 = -np.ones(tx_end-tx_start+1)
        if len(jn_start) > 0:
            Y0 = np.zeros(tx_end-tx_start+1)
            for c in jn_start:
                if tx_start <= c <= tx_end:
                    Y_type.append(2)
                    Y_idx.append(c-tx_start)
            for c in jn_end:
                if tx_start <= c <= tx_end:
                    Y_type.append(1)
                    Y_idx.append(c-tx_start)

    elif strand == '-':
        Y0 = -np.ones(tx_end-tx_start+1)

        if len(jn_start) > 0:
            Y0 = np.zeros(tx_end-tx_start+1)
            for c in jn_end:
                if tx_start <= c <= tx_end:
                    Y_type.append(2)
                    Y_idx.append(tx_end-c)
            for c in jn_start:
                if tx_start <= c <= tx_end:
                    Y_type.append(1)
                    Y_idx.append(tx_end-c)

    return jn_start,jn_end,Y_type, Y_idx

In [29]:
data_dir = '../Data'
if os.path.exists('{}/gencode_test.tsv'.format(data_dir)):
        os.remove('{}/gencode_test.tsv'.format(data_dir))
db = gffutils.FeatureDB(db_path)
test_chroms = {"chr1", "chr3", "chr5", "chr7", "chr9"}
genes = db.features_of_type('gene')
for gene in genes:
    chrom = gene[0]
    if chrom not in test_chroms:
        continue

    strand = gene[6]
    gene_start = gene[3]
    gene_end = gene[4]
    transcripts = db.children(gene, featuretype="transcript")
    for transcript in transcripts:
        transcript_id = transcript['transcript_id'][0]
        if transcript['transcript_type'][0]!='protein_coding':
            continue

        jn_start,jn_end,Y_type, Y_idx = getJunctions(db,transcript_id)
        
        tx_start = int(transcript[3])
        tx_end = int(transcript[4])

        name = '{}---{}.{}---{}.{}'.format(gene['gene_name'][0],gene['gene_id'][0],gene['gene_type'][0],transcript['transcript_id'][0],transcript['transcript_type'][0])
            
        if strand=='+':
            with open('{}/gencode_test.tsv'.format(data_dir), 'a') as the_file:
                the_file.write('{}\t{}\t{}\t{}\t{}\t{}\t{}\n'.format(name,chrom,strand,tx_start,tx_end,','.join([str(x) for x in jn_start]),','.join([str(x) for x in jn_end])))
        if strand=='-':
            with open('{}/gencode_test.tsv'.format(data_dir), 'a') as the_file:
                the_file.write('{}\t{}\t{}\t{}\t{}\t{}\t{}\n'.format(name,chrom,strand,tx_start,tx_end,','.join([str(x) for x in jn_end]),','.join([str(x) for x in jn_start])))



In [12]:
data_dir = '../Data'

CL_max=40000
# Maximum nucleotide context length (CL_max/2 on either side of the 
# position of interest)
# CL_max should be an even number
SL=5000

BATCH_SIZE = 1

setType = 'test'
annotation, transcriptToLabel, seqData = getData(data_dir, setType)

In [111]:
row = annotation.iloc[0]
chrom = row['chrom']
start = row['tx_start']
end = row['tx_end']
transcript_id = row['transcript']

seq = seqData[chrom][start:end]  # sparse matrix slice

Y_type, Y_idx = transcriptToLabel[transcript_id]

print(seqData)


{'chr1': <Compressed Sparse Row sparse matrix of dtype 'int8'
	with 103736881 stored elements and shape (248956422, 5)>, 'chr3': <Compressed Sparse Row sparse matrix of dtype 'int8'
	with 93687875 stored elements and shape (198295559, 5)>, 'chr5': <Compressed Sparse Row sparse matrix of dtype 'int8'
	with 64571606 stored elements and shape (181538259, 5)>, 'chr7': <Compressed Sparse Row sparse matrix of dtype 'int8'
	with 70446509 stored elements and shape (159345973, 5)>, 'chr9': <Compressed Sparse Row sparse matrix of dtype 'int8'
	with 47127602 stored elements and shape (138394717, 5)>}


In [51]:
print(annotation)

                                                   name chrom strand  \
0     FO538757.2---ENSG00000279928.1---ENST000006244...  chr1      +   
1     NOC2L---ENSG00000188976.10---ENST00000327044.6...  chr1      -   
2     KLHL17---ENSG00000187961.13---ENST00000338591....  chr1      +   
3     PLEKHN1---ENSG00000187583.10---ENST00000379407...  chr1      +   
4     PLEKHN1---ENSG00000187583.10---ENST00000379410...  chr1      +   
...                                                 ...   ...    ...   
8950  DPH7---ENSG00000148399.12---ENST00000277540.6-...  chr9      -   
8951  ZMYND19---ENSG00000165724.5---ENST00000298585....  chr9      -   
8952  ARRDC1---ENSG00000197070.13---ENST00000371421....  chr9      +   
8953  EHMT1---ENSG00000181090.18---ENST00000462484.5...  chr9      +   
8954  CACNA1B---ENSG00000148408.12---ENST00000277549...  chr9      +   

       tx_start     tx_end       transcript             gene  
0        182393     184158  ENST00000624431  ENSG00000279928  
1        

In [13]:
train_gene, test_gene = train_test_split(annotation['gene'].drop_duplicates(),test_size=.001,random_state=435)
annotation_train = annotation[annotation['gene'].isin(train_gene)]
annotation_test = annotation[annotation['gene'].isin(test_gene)]

In [53]:
print(annotation_test)

                                                   name chrom strand  \
38    MIB2---ENSG00000197530.12---ENST00000355826.9-...  chr1      +   
39    MIB2---ENSG00000197530.12---ENST00000505820.6-...  chr1      +   
40    MIB2---ENSG00000197530.12---ENST00000518681.5-...  chr1      +   
41    MIB2---ENSG00000197530.12---ENST00000520777.5-...  chr1      +   
316   ARHGEF19---ENSG00000142632.16---ENST0000027074...  chr1      -   
...                                                 ...   ...    ...   
8604  MAPKAP1---ENSG00000119487.16---ENST00000373511...  chr9      -   
8605  MAPKAP1---ENSG00000119487.16---ENST00000394060...  chr9      -   
8606  MAPKAP1---ENSG00000119487.16---ENST00000468896...  chr9      -   
8704  LRRC8A---ENSG00000136802.11---ENST00000372599....  chr9      +   
8705  LRRC8A---ENSG00000136802.11---ENST00000372600....  chr9      +   

       tx_start     tx_end       transcript             gene  
38      1615421    1630610  ENST00000355826  ENSG00000197530  
39      1

In [14]:
temp = 1
n_models = 10
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model_m = SpliceFormer(CL_max,bn_momentum=0.01/1,depth=4,heads=4,n_transformer_blocks=2,determenistic=True)
model_m.apply(keras_init)
model_m = model_m.to(device)

if torch.cuda.device_count() > 1:
    model_m = nn.DataParallel(model_m)
model_m = nn.DataParallel(model_m)
output_class_labels = ['Null', 'Acceptor', 'Donor']

#for output_class in [1,2]:
models = [copy.deepcopy(model_m) for i in range(n_models)]
[model.load_state_dict(torch.load('../Results/PyTorch_Models/transformer_encoder_40k_171022_{}'.format(i),map_location=device)) for i,model in enumerate(models)]
#nr = [0,2,3]
#[model.load_state_dict(torch.load('../Results/PyTorch_Models/transformer_encoder_40k_201221_{}'.format(nr[i]))) for i,model in enumerate(models)]
#chunkSize = num_idx/10
for model in models:
    model.eval()

Y_true_acceptor, Y_pred_acceptor = [],[]
Y_true_donor, Y_pred_donor = [],[]
test_dataset = spliceDataset(getDataPointListFull(annotation_test,transcriptToLabel,SL,CL_max,shift=SL))
test_dataset.seqData = seqData
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=10)

outpath = '../Data/TESTING_CSV.csv'
batch = []
flush = 3
if not os.path.exists(outpath):
    with open(outpath, 'wt') as f:
        f.write("Y_true_acceptor,Y_pred_acceptor,Y_true_donor,Y_pred_donor\n")

#targets_list = []
#outputs_list = []
ce_2d = []
for (batch_features ,targets) in tqdm(test_loader):
    batch_features = batch_features.type(torch.FloatTensor).to(device)
    targets = targets.to(torch.float32).to("mps")[:,:,CL_max//2:-CL_max//2]
    outputs = ([models[i](batch_features)[0].detach() for i in range(n_models)])
    #outputs = (outputs[0]+outputs[1]+outputs[2]+outputs[3]+outputs[4])/n_models
    outputs = torch.stack(outputs)
    outputs = torch.mean(outputs,dim=0)
    #outputs = odds_gmean(outputs)
    #targets_list.extend(targets.unsqueeze(0))
    #outputs_list.extend(outputs.unsqueeze(0))

    targets = torch.transpose(targets,1,2).cpu().numpy()
    outputs = torch.transpose(outputs,1,2).cpu().numpy()
    ce_2d.append(cross_entropy_2d(targets,outputs))

    is_expr = (targets.sum(axis=(1,2)) >= 1)
    Y_true_acceptor = targets[is_expr, :, 1].flatten()
    Y_true_donor = targets[is_expr, :, 2].flatten()
    Y_pred_acceptor = outputs[is_expr, :, 1].flatten()
    Y_pred_donor = outputs[is_expr, :, 2].flatten()

    df = pd.DataFrame({'Y_true_acceptor':Y_true_acceptor,'Y_pred_acceptor':Y_pred_acceptor,'Y_true_donor':Y_true_donor,'Y_pred_donor':Y_pred_donor})
    buffer.append(df)
    if len(buffer) >= flush:
        pd.concat(buffer).to_csv(outpath, mode='a', index=False, header=False)
        buffer = []
if buffer:
    pd.concat(buffer).to_csv(outpath, mode='a', index=False, header=False)



  0%|          | 0/19 [00:00<?, ?it/s]

In [58]:
mean_ce = np.mean(ce_2d)
print('Cross entropy = {}'.format(mean_ce))
Y_true_acceptor, Y_pred_acceptor,Y_true_donor, Y_pred_donor = np.array(Y_true_acceptor), np.array(Y_pred_acceptor),np.array(Y_true_donor), np.array(Y_pred_donor)
print("\n\033[1m{}:\033[0m".format('Acceptor'))
acceptor_val_results = print_topl_statistics(Y_true_acceptor, Y_pred_acceptor)
print("\n\033[1m{}:\033[0m".format('Donor'))
donor_val_results =print_topl_statistics(Y_true_donor, Y_pred_donor)

Cross entropy = 0.00011905628343811259

Acceptor:
0.9751	0.9276	0.9887	0.9943	0.9534	0.9824	0.6519	0.0025	0.0003	820	884.0	884

Donor:
0.9548	0.9344	0.9898	0.9955	0.9489	0.9843	0.6147	0.0023	0.0003	826	884.0	884


In [59]:
df = pd.DataFrame({'Y_true_acceptor':Y_true_acceptor,'Y_pred_acceptor':Y_pred_acceptor,'Y_true_donor':Y_true_donor,'Y_pred_donor':Y_pred_donor})
df.to_csv('../Data/transformer_40k_test_subset_predictions_180625.gz',index=False)

In [91]:
df = pd.read_csv('../Data/spliceai_10k_test_ensembl_predictions_300625.csv.gz')
df

,Y_true_acceptor,Y_pred_acceptor,Y_true_donor,Y_pred_donor
0,0.0,5.405732e-07,0.0,9.749649e-06
1,0.0,4.137088e-07,0.0,2.795269e-06
2,0.0,3.342412e-07,0.0,2.451711e-05
3,0.0,6.124780e-07,0.0,7.294009e-06
4,0.0,8.705280e-07,0.0,2.021087e-06
...,...,...,...,...
664939995,0.0,1.134575e-06,0.0,1.238303e-06
664939996,0.0,1.777195e-06,0.0,9.238735e-07
664939997,0.0,1.403097e-06,0.0,9.033723e-07
664939998,0.0,1.227144e-06,0.0,5.584995e-07


In [93]:

device = torch.device("cpu")
Y_true_acceptor = torch.as_tensor(df['Y_true_acceptor'].values, dtype=torch.int8).to(device)
Y_pred_acceptor = torch.as_tensor(df['Y_pred_acceptor'].values, dtype=torch.float32).to(device)
Y_true_donor = torch.as_tensor(df['Y_true_donor'].values, dtype=torch.int8).to(device)
Y_pred_donor = torch.as_tensor(df['Y_pred_donor'].values, dtype=torch.float32).to(device)


# Compute scores
ap_score = calculate_ap(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor, device, device)
topk_score = calculate_topk(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor)

print(f"Average Precision: {ap_score:.4f}")
print(f"Top-k Accuracy: {topk_score:.4f}")

Average Precision: 0.9933
Top-k Accuracy: 0.9706


In [87]:
import numpy as np
import h5py
import pandas as pd
import os
import gzip
num_iterations = 10000
samples_per_batch = 100
buffer = []
flush = 500
first_write = True
outpath = '../Data/TESTING_h5py.h5'
outpath_csv = '../Data/TESTING_csv.csv'
outpath_csvgz = '../Data/TESTING_csv.csv.gz'
if os.path.exists(outpath):
    os.remove(outpath)
if os.path.exists(outpath_csv):
    os.remove(outpath_csv)
if os.path.exists(outpath_csvgz):
    os.remove(outpath_csvgz)

# with h5py.File(outpath, 'w') as f:
#     f.create_dataset('Y_true_acceptor', data=Y_true_acceptor)
#     f.create_dataset('Y_true_donor', data=Y_true_donor)
#     f.create_dataset('Y_pred_acceptor', data=Y_pred_acceptor)
#     f.create_dataset('Y_pred_donor', data=Y_pred_donor)
store = pd.HDFStore(outpath)
for i in tqdm(range(num_iterations)):
    # Simulate binary ground truth labels (0 or 1)
    Y_true_acceptor = np.random.randint(0, 2, size=samples_per_batch).astype(np.float32)
    Y_true_donor = np.random.randint(0, 2, size=samples_per_batch).astype(np.float32)

    Y_pred_acceptor = np.random.rand(samples_per_batch).astype(np.float32)
    Y_pred_donor = np.random.rand(samples_per_batch).astype(np.float32)
    df = pd.DataFrame({'Y_true_acceptor':Y_true_acceptor,'Y_pred_acceptor':Y_pred_acceptor,'Y_true_donor':Y_true_donor,'Y_pred_donor':Y_pred_donor})
    # df.to_hdf(outpath,key='table',append=True,mode='a')
    # print(pd.read_hdf(outpath, mode='r'))
    store.append('data',df,format='table',index=False)
    if i==0:
        df.to_csv(outpath_csv,mode='a',index=False,header=True)
    else:
        df.to_csv(outpath_csv,mode='a',index=False,header=False)
    # with gzip.open(outpath_csvgz, 'a') as compressed_file:
    #     compressed_file.write(df.to_csv(index=False,header=False).encode())
    buffer.append(df)
    if len(buffer) >= flush:
        pd.concat(buffer).to_csv(outpath_csvgz, mode='a', index=False, header=first_write, compression='gzip')
        buffer = []
        first_write = False
if buffer:
    pd.concat(buffer).to_csv(outpath_csvgz, mode='a', index=False, header=False, compression='gzip')
epic = store['data']


In [89]:
csvgz = (pd.read_csv(outpath_csvgz))
csv = (pd.read_csv(outpath_csv))
print(csv)
print(csvgz)
print(csv.equals(csvgz))

        Y_true_acceptor  Y_pred_acceptor  Y_true_donor  Y_pred_donor
0                   1.0         0.076086           0.0      0.965291
1                   0.0         0.477069           0.0      0.517628
2                   1.0         0.730845           0.0      0.179430
3                   0.0         0.037817           0.0      0.016828
4                   1.0         0.334563           1.0      0.609435
...                 ...              ...           ...           ...
999995              1.0         0.465793           0.0      0.660015
999996              1.0         0.469952           1.0      0.784173
999997              1.0         0.492508           0.0      0.668619
999998              0.0         0.416624           1.0      0.991934
999999              0.0         0.036122           0.0      0.377228

[1000000 rows x 4 columns]
        Y_true_acceptor  Y_pred_acceptor  Y_true_donor  Y_pred_donor
0                   1.0         0.076086           0.0      0.965291
1     

In [30]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
Y_true_acceptor = torch.as_tensor(epic['Y_true_acceptor'].values, dtype=torch.int8).to(device)
Y_pred_acceptor = torch.as_tensor(epic['Y_pred_acceptor'].values, dtype=torch.float32).to(device)
Y_true_donor = torch.as_tensor(epic['Y_true_donor'].values, dtype=torch.int8).to(device)
Y_pred_donor = torch.as_tensor(epic['Y_pred_donor'].values, dtype=torch.float32).to(device)

# Compute scores
ap_score = calculate_ap(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor, device, device)
topk_score = calculate_topk(Y_true_acceptor, Y_pred_acceptor, Y_true_donor, Y_pred_donor)

print(f"Average Precision: {ap_score:.4f}")
print(f"Top-k Accuracy: {topk_score:.4f}")

Average Precision: 0.4654
Top-k Accuracy: 0.4669
